# 📖 Notebook 2: Pods & Deployments — Running Your Applications

Now that you have a cluster, it's time to run real workloads.
In this notebook, you'll create a pod, package sample services into your cluster, and learn how Deployments keep your applications healthy.

## Learning Objectives

By the end of this notebook, you'll be able to:
- Explain what a pod is and why it is the smallest deployable Kubernetes unit
- Create a pod from YAML
- Build sample application images directly into minikube
- Explain what a Deployment does for you
- Scale a Deployment to multiple replicas
- Perform a rolling update and watch the rollout
- Roll back to the previous version if needed
- Add resource requests and limits to a running workload


## 🛠️ Setup

Make sure Notebook 1 is complete before starting this one.
You need a running minikube cluster because every exercise in this notebook talks to Kubernetes.

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → 'Reload Window'.

### Exercise

Verify that the cluster is running and that `kubectl` can see your node.


In [ ]:
!minikube status
!kubectl get nodes


## 🧱 Build the Sample Application Images

The lab folder includes three small sample services:
- `user-service`
- `order-service`
- `api-gateway`

When you use `minikube image build`, the image gets built inside minikube's container environment.
That means Kubernetes can use the image immediately without pushing it to Docker Hub.

### Exercise

Build the sample images into minikube.


In [ ]:
!minikube image build -t k8s-lab/user-service:latest ../apps/user-service/
!minikube image build -t k8s-lab/order-service:latest ../apps/order-service/
!minikube image build -t k8s-lab/api-gateway:latest ../apps/api-gateway/


## 📦 What is a Pod?

A **pod** is the smallest thing Kubernetes schedules.
It usually contains one application container, but it can hold multiple containers that need to stay together.

Analogy: if a **container** is one worker, a **pod** is that worker plus their desk, badge, and local tools.
Kubernetes does not schedule the worker alone; it schedules the whole desk setup as one unit.

```text
┌──────────────────────────────┐
│            Pod               │
│                              │
│  ┌────────────────────────┐  │
│  │ user-service container │  │
│  └────────────────────────┘  │
│  shared IP + shared volumes  │
└──────────────────────────────┘
```

We'll save our YAML in the notebook folder so you can inspect it later.

### Exercise

Write a pod manifest.


In [ ]:
%%writefile ./pod.yaml
apiVersion: v1
kind: Pod
metadata:
  name: hello-pod
  labels:
    app: hello-pod
spec:
  containers:
    - name: hello-pod
      image: nginx:1.27
      ports:
        - containerPort: 80


In [ ]:
!kubectl apply -f ./pod.yaml
!kubectl wait --for=condition=Ready pod/hello-pod --timeout=120s
!kubectl get pods
!kubectl describe pod hello-pod
!kubectl logs hello-pod


## 🚚 What is a Deployment?

A **Deployment** manages pods for you.
You tell Kubernetes the desired state — for example, "keep 3 copies of `user-service` running" — and the Deployment keeps trying to make that true.

This gives you three huge benefits:
- **Self-healing**: if a pod dies, Kubernetes creates a replacement
- **Scaling**: you can raise or lower the replica count
- **Rolling updates**: Kubernetes replaces old pods gradually instead of all at once

```text
Deployment
   │
   ▼
ReplicaSet
   │
   ├── Pod 1
   ├── Pod 2
   └── Pod 3
```

### Exercise

Write a Deployment manifest for `user-service`.


In [ ]:
%%writefile ./user-service-deployment.yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: user-service
spec:
  replicas: 1
  selector:
    matchLabels:
      app: user-service
  template:
    metadata:
      labels:
        app: user-service
    spec:
      containers:
        - name: user-service
          image: k8s-lab/user-service:latest
          ports:
            - containerPort: 8001


In [ ]:
!kubectl apply -f ./user-service-deployment.yaml
!kubectl rollout status deployment/user-service
!kubectl get deployments
!kubectl get pods -l app=user-service -o wide


## 📈 Scaling Replicas

If one pod can handle some traffic, then multiple replicas can handle more traffic and give you better availability.
Scaling a Deployment changes the desired number of pods.

### Exercise

Scale `user-service` from 1 replica to 3 replicas.


In [ ]:
!kubectl scale deployment user-service --replicas=3
!kubectl get deployments
!kubectl get pods -l app=user-service


## 🔄 Rolling Updates

A rolling update changes a Deployment without stopping everything at once.
Kubernetes gradually creates new pods and removes old ones.

For a safe demo, we'll build the same app code with a new image tag called `v2`.
Even if the code is unchanged, the new tag still lets you see the rollout process.

### Exercise

Build a new image tag, update the Deployment, and watch Kubernetes finish the rollout.


In [ ]:
!minikube image build -t k8s-lab/user-service:v2 ../apps/user-service/
!kubectl set image deployment/user-service user-service=k8s-lab/user-service:v2
!kubectl rollout status deployment/user-service
!kubectl get pods -l app=user-service


## ⏪ Rollback

If a rollout goes badly, Kubernetes can return the Deployment to the previous version.
This is one of the best beginner examples of declarative operations: Kubernetes remembers rollout history for Deployments.

### Exercise

Undo the last rollout and confirm the Deployment becomes healthy again.


In [ ]:
!kubectl rollout undo deployment/user-service
!kubectl rollout status deployment/user-service
!kubectl describe deployment user-service


## 📏 Resource Requests and Limits

Kubernetes needs to know roughly how much CPU and memory your app expects.

- A **request** says: "please reserve at least this much for me"
- A **limit** says: "do not let me go above this amount"

```text
container
  request: guaranteed starting space
  limit:   hard ceiling
```

If you skip these values in real systems, scheduling and stability get harder.

### Exercise

Add CPU and memory requests and limits to the Deployment.


In [ ]:
!kubectl set resources deployment user-service --requests=cpu=100m,memory=128Mi --limits=cpu=250m,memory=256Mi
!kubectl rollout status deployment/user-service
!kubectl describe deployment user-service


## 🧹 Clean Up

When a lab ends, remove the resources you created so the next exercise starts clean.

### Exercise

Delete the pod, the Deployment, and the local YAML files created by this notebook.


In [ ]:
!kubectl delete deployment user-service --ignore-not-found
!kubectl delete pod hello-pod --ignore-not-found
!rm -f ./pod.yaml ./user-service-deployment.yaml


## 🎓 What You Learned

In this notebook, you:
- Created a pod from YAML
- Built sample images directly into minikube
- Created a Deployment for `user-service`
- Scaled replicas up to 3
- Performed a rolling update to a new image tag
- Used a rollback to return to the previous version
- Added CPU and memory requests and limits

You now understand the difference between **one pod** and a **managed application rollout**.
That is the foundation for Services, networking, and production-style Kubernetes operations.
